# ClinTrialQA — Automated Benchmark Pipeline

Evaluates epistemic calibration of AI models over real clinical trial evidence.


### Before running
Add secrets via the  icon in the left sidebar:
- `GROQ_API_KEY` — console.groq.com (free, no credit card)
- `ENTREZ_EMAIL` — any email (NCBI requires it, free)
- `NCBI_API_KEY` — optional, ncbi.nlm.nih.gov/account (10× rate limit)


In [ ]:
# ══ Cell 1: Install & imports ══════════════════════════════════════════════════
!pip install groq biopython requests tqdm -q

import csv, json, os, re, time, shutil
from pathlib import Path
from typing import Optional

import groq as groq_lib
import pandas as pd
from Bio import Entrez
from google.colab import userdata, files, drive
from tqdm.notebook import tqdm
from IPython.display import display

print('✓ All packages loaded')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.1 MB/s eta 0:00:00
✓ All packages loaded


In [ ]:
# ══ Cell 2: Credentials & Google Drive mount ═══════════════════════════════════

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['ENTREZ_EMAIL'] = userdata.get('ENTREZ_EMAIL')
try:
    os.environ['NCBI_API_KEY'] = userdata.get('NCBI_API_KEY')
except Exception:
    pass

Entrez.email = os.environ['ENTREZ_EMAIL']
ncbi_key = os.environ.get('NCBI_API_KEY')
if ncbi_key:
    Entrez.api_key = ncbi_key
    print('✓ NCBI API key set (10 req/s rate limit)')
else:
    print('ℹ  No NCBI API key — using 3 req/s (add NCBI_API_KEY secret for faster runs)')

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/ClinTrialQA')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'✓ Drive mounted → results will also save to {DRIVE_DIR}')

client = groq_lib.Groq(api_key=os.environ.get('GROQ_API_KEY'))
print('✓ Groq client ready')


ℹ  No NCBI API key — using 3 req/s (add NCBI_API_KEY secret for faster runs)
Mounted at /content/drive
✓ Drive mounted → results will also save to /content/drive/MyDrive/ClinTrialQA
✓ Groq client ready


In [ ]:
# ══ Cell 3: Configuration ══════════════════════════════════════════════════════

DOMAINS = ['Cardiology', 'Endocrinology']   # medical domains

# Desired FINAL accepted dataset size.
# The pipeline will generate more candidate questions than this, then stop once enough valid items pass validation.
N_ITEMS = 10

# Candidate buffer: PubMed retrieval/classification/reference generation can reject many candidates.

CANDIDATE_MULTIPLIER = 2
MAX_CANDIDATES = max(N_ITEMS, N_ITEMS * CANDIDATE_MULTIPLIER)

# Models to evaluate. Use llama-4-scout by default because your Groq limit is 30K TPM.
# Keep llama-3.3-70b-versatile disabled by default because it only has 12K TPM and can rate-limit easily.
MODELS  = ['meta-llama/llama-4-scout-17b-16e-instruct']
#MODELS = ['llama-3.3-70b-versatile']

# Internal models
# meta-llama/llama-4-scout-17b-16e-instruct : 30 RPM / 30K TPM / 500K TPD — safer for generation + evaluation
# llama-3.1-8b-instant                       : 30 RPM /  6K TPM / 500K TPD — good for short classification calls
# llama-3.3-70b-versatile                    : 30 RPM / 12K TPM / 100K TPD — high quality, but use only with longer waits
GENERATOR_MODEL  = 'meta-llama/llama-4-scout-17b-16e-instruct'
CLASSIFIER_MODEL = 'llama-3.1-8b-instant'
EVALUATOR_MODEL  = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Optional high-quality evaluator setting. Uncomment only if you are okay with much slower calls.
# EVALUATOR_MODEL = 'llama-3.3-70b-versatile'

DATASET_PATH = Path('dataset/clintrialqa_v1.json')
RESULTS_DIR  = Path('evaluation')

print('Configuration:')
print(f'  Domains:              {DOMAINS}')
print(f'  Target final items:   {N_ITEMS}')
print(f'  Max candidate items:  {MAX_CANDIDATES}  ({CANDIDATE_MULTIPLIER}x buffer)')
print(f'  Models:               {MODELS}')
print(f'  Classifier:           {CLASSIFIER_MODEL}')
print(f'  Generator:            {GENERATOR_MODEL}')
print(f'  Evaluator:            {EVALUATOR_MODEL}')


Configuration:
  Domains:              ['Cardiology', 'Endocrinology']
  Target final items:   10
  Max candidate items:  20  (2x buffer)
  Models:               ['meta-llama/llama-4-scout-17b-16e-instruct']
  Classifier:           llama-3.1-8b-instant
  Generator:            meta-llama/llama-4-scout-17b-16e-instruct
  Evaluator:            meta-llama/llama-4-scout-17b-16e-instruct


In [ ]:
# ══ Cell 4: Pipeline functions ═════════════════════════════════════════════════

# Groq limits from your current account page. Used for adaptive throttling.
GROQ_MODEL_LIMITS = {
    'llama-3.1-8b-instant': {'rpm': 30, 'tpm': 6000},
    'llama-3.3-70b-versatile': {'rpm': 30, 'tpm': 12000},
    'meta-llama/llama-4-scout-17b-16e-instruct': {'rpm': 30, 'tpm': 30000},
    'openai/gpt-oss-120b': {'rpm': 30, 'tpm': 8000},
    'openai/gpt-oss-20b': {'rpm': 30, 'tpm': 8000},
    'qwen/qwen3-32b': {'rpm': 60, 'tpm': 6000},
}

TOKEN_SAFETY_FACTOR = 1.5
_last_llm_call_at = {}

def _estimate_tokens(text):
    """Rough token estimate without adding a tokenizer dependency."""
    return max(1, len(str(text)) // 4)

def _required_delay_seconds(model, estimated_tokens):
    limits = GROQ_MODEL_LIMITS.get(model, {'rpm': 30, 'tpm': 6000})
    rpm_delay = 60.0 / limits['rpm']
    tpm_delay = 60.0 * (estimated_tokens * TOKEN_SAFETY_FACTOR) / limits['tpm']
    return max(rpm_delay, tpm_delay)

def _wait_before_request(model, estimated_tokens):
    needed = _required_delay_seconds(model, estimated_tokens)
    last = _last_llm_call_at.get(model, 0)
    elapsed = time.time() - last
    if elapsed < needed:
        wait = needed - elapsed
        print(f'    Throttling {model}: waiting {wait:.1f}s to stay under RPM/TPM limits...')
        time.sleep(wait)
    _last_llm_call_at[model] = time.time()

def _retry_after_seconds(error):
    """Try to extract Groq's suggested retry-after delay from headers or error text."""
    response = getattr(error, 'response', None)
    headers = getattr(response, 'headers', {}) if response is not None else {}
    for key in ('retry-after', 'Retry-After'):
        if key in headers:
            try:
                return float(headers[key]) + 1.0
            except Exception:
                pass

    msg = str(error)
    # Groq messages often include phrases like "Please try again in 7.2s" or "1m2.3s".
    m = re.search(r'try again in\s+([^\n\.]+(?:\.\d+)?[a-z]*)', msg, flags=re.I)
    if not m:
        return None
    chunk = m.group(1).lower()
    total = 0.0
    for value, unit in re.findall(r'(\d+(?:\.\d+)?)(ms|s|m|h)', chunk):
        value = float(value)
        if unit == 'ms':
            total += value / 1000.0
        elif unit == 's':
            total += value
        elif unit == 'm':
            total += value * 60.0
        elif unit == 'h':
            total += value * 3600.0
    return total + 1.0 if total > 0 else None

def call_llm(prompt, model, max_tokens=600, retries=5, temperature=0.1):
    """
    Call Groq with adaptive RPM/TPM throttling and useful final errors.
    This avoids the old problem where all failures became only:
    RuntimeError: All 5 retries failed for model ...
    """
    last_error = None
    estimated_tokens = _estimate_tokens(prompt) + max_tokens

    for attempt in range(retries):
        try:
            _wait_before_request(model, estimated_tokens)
            resp = client.chat.completions.create(
                model=model,
                max_tokens=max_tokens,
                temperature=temperature,
                messages=[{'role': 'user', 'content': prompt}],
            )
            text = resp.choices[0].message.content
            if not text or not text.strip():
                raise ValueError('Empty response from model')
            return text

        except groq_lib.RateLimitError as e:
            last_error = e
            suggested = _retry_after_seconds(e)
            fallback = max(_required_delay_seconds(model, estimated_tokens), 15 * (2 ** attempt))
            wait = suggested if suggested is not None else fallback
            print(f'    Rate limited on {model} — waiting {wait:.1f}s...')
            print(f'    Last rate-limit message: {str(e)[:220]}')
            time.sleep(wait)

        except ValueError as e:
            last_error = e
            wait = min(120, 10 * (2 ** attempt))
            print(f'    Empty/invalid response from {model} — retrying in {wait}s...')
            time.sleep(wait)

        except Exception as e:
            last_error = e
            wait = min(120, 5 * (2 ** attempt))
            print(f'    Error from {model}: {type(e).__name__}: {str(e)[:220]}')
            if attempt == retries - 1:
                break
            print(f'    Retrying in {wait}s...')
            time.sleep(wait)

    raise RuntimeError(
        f'All {retries} retries failed for model {model}. '
        f'Last error: {type(last_error).__name__}: {last_error}'
    )

def parse_json(text):
    """Robustly extract JSON from a model response.

    Note: if the model output is genuinely truncated, no parser can safely recover
    the missing closing objects. In that case, reduce batch size or increase
    max_tokens in the caller.
    """
    if not text or not text.strip():
        raise ValueError('Empty response — cannot parse JSON')

    clean = text.strip()

    # Remove common Markdown fences.
    clean = re.sub(r'^\s*```(?:json)?\s*', '', clean, flags=re.IGNORECASE)
    clean = re.sub(r'\s*```\s*$', '', clean).strip()

    try:
        return json.loads(clean)
    except json.JSONDecodeError as e:
        first_error = e

    # Try extracting the largest JSON object from surrounding text.
    match = re.search(r'(\{[\s\S]*\})', clean)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    preview = clean[:1000]
    raise ValueError(
        'No valid JSON found. This often means the LLM response was truncated. '
        f'Raw length={len(clean)} chars. First JSON error: {first_error}. '
        f'Response preview:\n{preview}'
    )

# ── Phase 0: Generate questions ───────────────────────────────────────────────

def _normalize_question(q, domain, target_tier):
    """Keep candidate-question objects in the schema expected by later cells."""
    return {
        'id': str(q.get('id', '')),
        'domain': str(q.get('domain', domain)).strip() or domain,
        'question': str(q.get('question', '')).strip(),
        'target_tier': str(q.get('target_tier', target_tier)).strip().upper() or target_tier,
        'pubmed_queries': [
            str(x).strip() for x in (q.get('pubmed_queries') or [])[:2] if str(x).strip()
        ],
        'irrelevant_query': str(q.get('irrelevant_query', '')).strip()
    }

def _valid_question_spec(q):
    return (
        q.get('domain')
        and q.get('question')
        and q.get('target_tier') in {'SUPPORTED', 'MIXED', 'WEAK', 'CONTRADICTED', 'INSUFFICIENT'}
        and isinstance(q.get('pubmed_queries'), list)
        and len(q.get('pubmed_queries')) >= 2
        and q.get('irrelevant_query')
    )

def generate_questions(domains, n_items, batch_size=10):
    """Generate candidate question specs in small JSON batches.

    Why batched: asking for 60 full question objects in one call often exceeds
    max_tokens and returns cut-off JSON. Batching keeps each response parseable.
    """
    tiers = ['SUPPORTED', 'MIXED', 'WEAK', 'CONTRADICTED', 'INSUFFICIENT']
    questions = []
    batch_no = 0

    while len(questions) < n_items:
        batch_no += 1
        remaining = n_items - len(questions)
        k = min(batch_size, remaining)

        # Rotate domains and tiers for balanced coverage.
        start = len(questions)
        requested = []
        for j in range(k):
            requested.append({
                'domain': domains[(start + j) % len(domains)],
                'target_tier': tiers[(start + j) % len(tiers)]
            })

        requested_text = '\n'.join(
            f'- {idx+1}. domain={r["domain"]}, target_tier={r["target_tier"]}'
            for idx, r in enumerate(requested)
        )

        prompt = (
            f'Design exactly {k} clinical research question specifications.\n\n'
            f'Use these exact domain/tier assignments:\n{requested_text}\n\n'
            f'Requirements for each object:\n'
            f'- question: a clinically answerable PICO-style question about intervention/exposure and outcome\n'
            f'- 2 PubMed search queries for relevant RCTs, meta-analyses, or clinical trial evidence\n'
            f'- 1 irrelevant_query likely to retrieve a wrong population, wrong outcome, or wrong study design\n'
            f'- Avoid duplicate questions from this run. Existing question topics:\n'
            f'{json.dumps([q["question"] for q in questions[-20:]], ensure_ascii=False)}\n\n'
            f'Tier meanings:\n'
            f'  SUPPORTED: 2-3 relevant RCT/meta-analysis abstracts support the claim\n'
            f'  MIXED: relevant evidence both supports and contradicts\n'
            f'  WEAK: one relevant source supports but stronger/more evidence contradicts\n'
            f'  CONTRADICTED: 2-3 relevant RCT/meta-analysis abstracts contradict the claim\n'
            f'  INSUFFICIENT: no direct RCT evidence for this exact question\n\n'
            f'Return ONLY valid JSON with this exact schema, no Markdown:\n'
            '{"questions":[{"id":"CLQ-001","domain":"<domain>","question":"<question>",'
            '"target_tier":"<tier>","pubmed_queries":["<query1>","<query2>"],"irrelevant_query":"<query>"}]}'
        )

        print(f'  Generating candidate batch {batch_no}: {k} questions ({len(questions)+k}/{n_items})...')
        raw = call_llm(prompt, GENERATOR_MODEL, max_tokens=2200, retries=5, temperature=0.2)
        data = parse_json(raw)
        batch = data.get('questions', [])

        for j, raw_q in enumerate(batch):
            assignment = requested[min(j, len(requested)-1)]
            q = _normalize_question(raw_q, assignment['domain'], assignment['target_tier'])
            if _valid_question_spec(q):
                questions.append(q)
            else:
                print(f'    Skipping malformed generated question in batch {batch_no}: {q}')

        # Safety valve: if a batch under-produces, keep going with later batches.
        if not batch:
            raise ValueError('Generator returned zero questions for a batch; cannot continue.')

    questions = questions[:n_items]
    for i, q in enumerate(questions):
        q['id'] = f'CLQ-{i+1:03d}'
    return questions

# ── Phase 1-2: PubMed search + fetch ─────────────────────────────────────────

def search_pubmed(query, max_results=8, rct=True):
    full = query
    if rct:
        full += ' AND (Clinical Trial[pt] OR Randomized Controlled Trial[pt] OR Meta-Analysis[pt])'
    try:
        h = Entrez.esearch(db='pubmed', term=full, retmax=max_results, sort='relevance')
        r = Entrez.read(h); h.close(); time.sleep(1.0)
        return r['IdList']
    except Exception as e:
        print(f'  Search error: {e}'); return []

def fetch_abstracts(pmids):
    if not pmids: return []
    try:
        h = Entrez.efetch(db='pubmed', id=','.join(pmids), rettype='abstract', retmode='xml')
        records = Entrez.read(h); h.close(); time.sleep(1.0)
    except Exception as e:
        print(f'  Fetch error: {e}'); return []
    out = []
    for article in records.get('PubmedArticle', []):
        try:
            med = article['MedlineCitation']; art = med['Article']
            pmid = str(med['PMID']); title = str(art.get('ArticleTitle', ''))
            texts = art.get('Abstract', {}).get('AbstractText', [])
            if isinstance(texts, list):
                parts = []
                for p in texts:
                    lbl = getattr(p, 'attributes', {}).get('Label', '')
                    parts.append(f'{lbl}: {str(p)}' if lbl else str(p))
                abstract = ' '.join(parts)
            else:
                abstract = str(texts)
            if not abstract.strip(): continue
            jrn  = str(art.get('Journal', {}).get('Title', ''))
            pub  = art.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {})
            year = str(pub.get('Year', pub.get('MedlineDate', '')))[:4]
            auths = art.get('AuthorList', [])
            name  = (f"{auths[0].get('LastName', '')} et al." if len(auths) > 1
                     else str(auths[0].get('LastName', '')) if auths else 'Unknown')
            out.append({'pmid': pmid, 'title': title, 'abstract': abstract,
                        'citation': f'{name}, {jrn} ({year})',
                        'url': f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/'})
        except Exception: continue
    return out

# ── Phase 3: Sequential classification ───────────────────────────────────────
# Sequential with adaptive RPM/TPM throttling built into call_llm.
# Uses llama-3.1-8b-instant for short classification calls.

CLASS_VALUES = {'SUPPORTING', 'CONTRADICTING', 'IRRELEVANT'}

CLASSIFIER_STOPWORDS = {
    'does', 'with', 'from', 'that', 'this', 'have', 'has', 'had', 'the', 'and', 'for',
    'are', 'was', 'were', 'been', 'being', 'into', 'about', 'after', 'before', 'among',
    'adults', 'adult', 'patients', 'patient', 'people', 'study', 'trial', 'trials',
    'effect', 'effects', 'risk', 'rate', 'rates', 'outcome', 'outcomes', 'therapy',
    'treatment', 'reduce', 'reduces', 'reduced', 'improve', 'improves', 'whether'
}

def _normalize_classification(label):
    """Map loose model labels into the three allowed tags."""
    text = str(label or '').upper().strip()
    if 'CONTRADICT' in text or 'REFUTE' in text or 'OPPOSE' in text or 'NO BENEFIT' in text:
        return 'CONTRADICTING'
    if 'SUPPORT' in text or 'POSITIVE' in text or 'BENEFIT' in text:
        return 'SUPPORTING'
    return 'IRRELEVANT'

def classify_one(question, ab):
    prompt = (
        'Classify this abstract for the benchmark question as exactly one of:\n'
        'SUPPORTING / CONTRADICTING / IRRELEVANT\n\n'
        'Definitions:\n'
        '- SUPPORTING = relevant evidence supports a positive answer to the question.\n'
        '- CONTRADICTING = relevant evidence shows no benefit, harm, non-inferiority of a shorter/de-escalated option, '
        'or otherwise opposes the claim in the question.\n'
        '- IRRELEVANT = wrong population, wrong intervention/comparator, wrong outcome, or not useful for the claim.\n\n'
        'Important: if the abstract studies the same disease area and directly compares the intervention, duration, '
        'or outcome in the question, it is RELEVANT even if its conclusion is negative. Negative relevant evidence '
        'should be CONTRADICTING, not IRRELEVANT.\n\n'
        f'Question: {question}\n'
        f'Abstract (PMID {ab["pmid"]}):\n{ab["abstract"][:1400]}\n\n'
        'Return ONLY JSON: {"classification":"SUPPORTING|CONTRADICTING|IRRELEVANT",'
        '"confidence":"HIGH|MEDIUM|LOW","reasoning":"<one sentence>"}'
    )
    try:
        raw = call_llm(prompt, CLASSIFIER_MODEL, max_tokens=180)
        res = parse_json(raw)
        return {
            'tag': _normalize_classification(res.get('classification', 'IRRELEVANT')),
            'confidence': res.get('confidence', 'LOW'),
            'reasoning': res.get('reasoning', '')
        }
    except Exception as e:
        return {'tag': 'IRRELEVANT', 'confidence': 'LOW', 'reasoning': str(e)}

def _keyword_overlap_suggests_relevance(question, abstract_text, min_overlap=3):
    """Cheap guardrail: re-check IRRELEVANT labels when the abstract looks topically close."""
    q_terms = {
        w for w in re.findall(r'[a-zA-Z][a-zA-Z0-9\-]{3,}', question.lower())
        if w not in CLASSIFIER_STOPWORDS
    }
    a_terms = set(re.findall(r'[a-zA-Z][a-zA-Z0-9\-]{3,}', abstract_text.lower()))
    overlap = q_terms & a_terms
    return len(overlap) >= min_overlap

def recheck_irrelevant_one(question, ab):
    """Second-pass check for likely false IRRELEVANT classifications."""
    prompt = (
        'The previous classifier marked this abstract as IRRELEVANT. Re-check carefully.\n\n'
        'Question:\n'
        f'{question}\n\n'
        f'Abstract title: {ab.get("title", "")}\n'
        f'Abstract PMID {ab.get("pmid", "")}:\n{ab.get("abstract", "")[:1600]}\n\n'
        'Rules:\n'
        '- If it directly studies the same clinical question, intervention/duration/comparator, population, or outcome, '
        'it is relevant.\n'
        '- Relevant evidence that supports the claim = SUPPORTING.\n'
        '- Relevant evidence that shows no benefit, a shorter strategy is comparable/superior, harm, or opposes the claim = CONTRADICTING.\n'
        '- Only use IRRELEVANT for clearly wrong population/intervention/outcome/design.\n\n'
        'Return ONLY JSON: {"classification":"SUPPORTING|CONTRADICTING|IRRELEVANT",'
        '"confidence":"HIGH|MEDIUM|LOW","reasoning":"<one sentence>"}'
    )
    try:
        raw = call_llm(prompt, CLASSIFIER_MODEL, max_tokens=180)
        res = parse_json(raw)
        return {
            'tag': _normalize_classification(res.get('classification', 'IRRELEVANT')),
            'confidence': res.get('confidence', 'LOW'),
            'reasoning': 'Second-pass relevance check: ' + str(res.get('reasoning', ''))
        }
    except Exception as e:
        return {
            'tag': ab.get('tag', 'IRRELEVANT'),
            'confidence': ab.get('cls_conf', 'LOW'),
            'reasoning': 'Second-pass relevance check failed: ' + str(e)
        }

def validate_irrelevant_tags(question, abstracts):
    """Re-check abstracts labeled IRRELEVANT when keyword overlap suggests they may actually be relevant."""
    for ab in abstracts:
        if ab.get('tag') == 'IRRELEVANT' and _keyword_overlap_suggests_relevance(question, ab.get('abstract', '')):
            res = recheck_irrelevant_one(question, ab)
            ab['tag'] = res['tag']
            ab['cls_conf'] = res['confidence']
            ab['cls_reason'] = res['reasoning']
    return abstracts

def classify_all(question, abstracts):
    """Sequential classification plus second-pass validation for likely false IRRELEVANT tags."""
    for ab in abstracts:
        res = classify_one(question, ab)
        ab['tag'] = res['tag']
        ab['cls_conf'] = res['confidence']
        ab['cls_reason'] = res['reasoning']
    return validate_irrelevant_tags(question, abstracts)

# ── Phase 4: Derive tier ──────────────────────────────────────────────────────

def derive_tier(abstracts):
    """Derive the final confidence tier from selected abstract labels."""
    s = sum(1 for a in abstracts if a['tag'] == 'SUPPORTING')
    c = sum(1 for a in abstracts if a['tag'] == 'CONTRADICTING')

    if s == 0 and c == 0:
        return 'INSUFFICIENT'
    if s >= 2 and c == 0:
        return 'SUPPORTED'
    if s >= 1 and c >= 1:
        return 'MIXED' if s >= c else 'WEAK'
    if s == 0 and c >= 2:
        return 'CONTRADICTED'
    if s == 1 and c == 0:
        return 'SUPPORTED'
    if s == 0 and c == 1:
        return 'CONTRADICTED'
    return 'MIXED'

# ── Phase 5: Reference answer ─────────────────────────────────────────────────

def _validate_reference_obj(obj):
    """Normalize and validate generated reference-answer JSON."""
    if not isinstance(obj, dict):
        raise ValueError('Reference answer is not a JSON object')

    obj.setdefault('reference_answer', '')
    obj.setdefault('required_caveats', [])
    obj.setdefault('annotation_rationale', '')

    if isinstance(obj['required_caveats'], str):
        obj['required_caveats'] = [obj['required_caveats']]

    obj['reference_answer'] = str(obj['reference_answer']).strip()
    obj['required_caveats'] = [str(x).strip() for x in obj['required_caveats'] if str(x).strip()]
    obj['annotation_rationale'] = str(obj['annotation_rationale']).strip()

    if not obj['reference_answer']:
        raise ValueError('Missing reference_answer')
    if obj['reference_answer'].startswith('[Error:'):
        raise ValueError('Reference answer contains an error placeholder')

    return obj

def _repair_reference_json(bad_output, question, tier):
    """Ask the generator to convert a non-JSON reference answer into valid JSON."""
    repair_prompt = (
        'Convert the following model output into valid JSON only. Do not add Markdown.\n'
        'Required JSON schema:\n'
        '{"reference_answer":"<complete reference answer>",'
        '"required_caveats":["<caveat 1>","<caveat 2>"],'
        '"annotation_rationale":"<brief rationale>"}\n\n'
        f'Question: {question}\nCorrect tier: {tier}\n\n'
        f'Model output to convert:\n{str(bad_output)[:3000]}'
    )
    fixed = call_llm(repair_prompt, GENERATOR_MODEL, max_tokens=900, temperature=0.0)
    return _validate_reference_obj(parse_json(fixed))

def _fallback_reference(question, abstracts, tier, error_msg=''):
    """Deterministic backup so dataset items never contain [Error: ...] as the reference answer."""
    supporting = [a for a in abstracts if a.get('tag') == 'SUPPORTING']
    contradicting = [a for a in abstracts if a.get('tag') == 'CONTRADICTING']
    irrelevant = [a for a in abstracts if a.get('tag') == 'IRRELEVANT']

    supp_labels = ', '.join(a['label'] for a in supporting) or 'none'
    cont_labels = ', '.join(a['label'] for a in contradicting) or 'none'
    irr_labels = ', '.join(a['label'] for a in irrelevant) or 'none'

    if tier == 'SUPPORTED':
        opening = 'The available evidence strongly supports a positive answer to the question.'
    elif tier == 'MIXED':
        opening = 'The available evidence is mixed, with both supportive and opposing or non-confirmatory findings.'
    elif tier == 'WEAK':
        opening = 'The available evidence is weak because contradictory or non-confirmatory evidence outweighs supportive evidence.'
    elif tier == 'CONTRADICTED':
        opening = 'The available evidence contradicts a positive answer to the question.'
    else:
        opening = 'There is no directly applicable evidence among the provided abstracts.'

    reference_answer = (
        f'{opening} Supporting abstracts: {supp_labels}. '
        f'Contradicting abstracts: {cont_labels}. '
        f'Irrelevant abstracts: {irr_labels}; these should not be used as direct evidence. '
        f'Confidence: {tier}.'
    )

    caveats = []
    if irrelevant:
        caveats.append(f'Exclude irrelevant abstracts {irr_labels} from the evidence synthesis.')
    if supporting and contradicting:
        caveats.append('Use calibrated language because the relevant evidence is mixed rather than uniformly positive or negative.')
    if error_msg:
        caveats.append('Reference answer was produced by deterministic fallback after JSON repair failed.')

    return {
        'reference_answer': reference_answer,
        'required_caveats': caveats,
        'annotation_rationale': (
            f'Tier derived from selected labels: {len(supporting)} supporting, '
            f'{len(contradicting)} contradicting, {len(irrelevant)} irrelevant.'
        )
    }

def generate_reference(question, abstracts, tier):
    ab_text = '\n\n'.join(
        f"{a['label']} [{a['tag']}] ({a['citation']}, PMID {a['pmid']}):\n{a['abstract'][:650]}"
        for a in abstracts
    )
    prompt = (
        'Write a reference answer for a clinical evidence synthesis benchmark.\n'
        'Return ONLY raw valid JSON. No Markdown. No headings. No code fences.\n\n'
        f'Question: {question}\nTier: {tier}\n\nAbstracts:\n{ab_text}\n\n'
        'Rules:\n'
        '1. Synthesize only SUPPORTING and CONTRADICTING abstracts.\n'
        '2. State IRRELEVANT abstracts are not applicable and briefly explain why.\n'
        '3. Language must match tier: SUPPORTED→"strongly supports", MIXED→"mixed", '
        'WEAK→"weak evidence", CONTRADICTED→"contradicts", INSUFFICIENT→"no relevant evidence".\n'
        f'4. End the reference answer with exactly: "Confidence: {tier}."\n\n'
        'Required JSON schema:\n'
        '{"reference_answer":"<answer>",'
        '"required_caveats":["<c1>","<c2>"],'
        '"annotation_rationale":"<text>"}'
    )

    last_error = None
    for attempt in range(2):
        try:
            raw = call_llm(prompt, GENERATOR_MODEL, max_tokens=900, temperature=0.0)
            return _validate_reference_obj(parse_json(raw))
        except Exception as e:
            last_error = e
            try:
                return _repair_reference_json(raw if 'raw' in locals() else str(e), question, tier)
            except Exception as repair_error:
                last_error = repair_error
                prompt += '\n\nPrevious output was invalid JSON. You must return ONLY the required JSON object.'

    return _fallback_reference(question, abstracts, tier, error_msg=str(last_error))

# ── Phase 5: Assemble one item ────────────────────────────────────────────────

def _select_abstracts_for_target(fetched, target):
    """
    Select abstracts for the target tier without relabeling spare abstracts incorrectly.

    Each item must still contain exactly 4 abstracts. For SUPPORTED and
    CONTRADICTED, accept both the original stronger pattern and the new
    2-relevant + 2-irrelevant pattern:
      SUPPORTED:    3 SUPPORTING + 1 IRRELEVANT, or 2 SUPPORTING + 2 IRRELEVANT
      CONTRADICTED: 3 CONTRADICTING + 1 IRRELEVANT, or 2 CONTRADICTING + 2 IRRELEVANT
    """
    tier_patterns = {
        'SUPPORTED': [
            (3, 0, 1),
            (2, 0, 2),
        ],
        'MIXED': [
            (2, 1, 1),
        ],
        'WEAK': [
            (1, 2, 1),
        ],
        'CONTRADICTED': [
            (0, 3, 1),
            (0, 2, 2),
        ],
        'INSUFFICIENT': [
            (0, 0, 4),
        ],
    }

    patterns = tier_patterns.get(target, [(2, 1, 1)])

    supp = [a for a in fetched if a['tag'] == 'SUPPORTING']
    cont = [a for a in fetched if a['tag'] == 'CONTRADICTING']
    irr = [a for a in fetched if a['tag'] == 'IRRELEVANT']

    # Try each valid evidence pattern in order. This allows 2 SUPPORTING +
    # 2 IRRELEVANT to count as SUPPORTED, and 2 CONTRADICTING + 2 IRRELEVANT
    # to count as CONTRADICTED, without forcing relevant abstracts to be relabeled.
    for ns, nc, ni in patterns:
        if len(supp) >= ns and len(cont) >= nc and len(irr) >= ni:
            selected = supp[:ns] + cont[:nc] + irr[:ni]
            if len(selected) == 4:
                return selected

    return None

def assemble_item(spec):
    iid, q, target = spec['id'], spec['question'], spec['target_tier']

    pmids = set()
    for query in spec['pubmed_queries']:
        pmids.update(search_pubmed(query))

    irrel = set(search_pubmed(spec['irrelevant_query'], rct=False)) - pmids
    all_pmids = list(pmids | irrel)

    if not all_pmids:
        return None

    fetched = fetch_abstracts(all_pmids)

    if len(fetched) < 4:
        return None

    fetched = classify_all(q, fetched)
    sel = _select_abstracts_for_target(fetched, target)

    if sel is None:
        print(f'  Skipping {iid}: not enough clean abstracts for target tier {target}')
        return None

    for i, ab in enumerate(sel):
        ab['label'] = f'A{i+1}'

    actual_tier = derive_tier(sel)
    ref = generate_reference(q, sel, actual_tier)

    # Final guardrail: never save an item with an error placeholder as its reference answer.
    if str(ref.get('reference_answer', '')).startswith('[Error:'):
        ref = _fallback_reference(q, sel, actual_tier, error_msg='reference generation returned error placeholder')

    return {
        'id': iid,
        'domain': spec['domain'],
        'target_tier': target,
        'confidence_tier': actual_tier,
        'question': q,
        'abstracts': [
            {
                'label': a['label'],
                'tag': a['tag'],
                'pmid': a['pmid'],
                'citation': a['citation'],
                'url': a['url'],
                'title': a['title'],
                'text': a['abstract'],
                'classification_confidence': a.get('cls_conf'),
                'classification_reasoning': a.get('cls_reason')
            }
            for a in sel
        ],
        'reference_answer': ref.get('reference_answer', ''),
        'required_caveats': ref.get('required_caveats', []),
        'annotation_rationale': ref.get('annotation_rationale', ''),
        'model_evaluations': []
    }

# ── Phase 6-7: Evaluate ───────────────────────────────────────────────────────

SYNTH = (
    'You are a clinical evidence synthesizer.\n'
    '1. State confidence: SUPPORTED / MIXED / WEAK / CONTRADICTED / INSUFFICIENT\n'
    '2. Identify any abstracts NOT relevant and exclude them\n'
    '3. Use hedging language calibrated to evidence strength\n\n'
    'Question: {question}\n\nAbstracts:\n{abstracts}\n\n'
    '**Confidence Level:** [tier]\n'
    '**Irrelevant Abstracts:** [labels or None]\n'
    '**Synthesis:** [answer]\n'
    '**Key Caveats:** [limitations]'
)

SCORER = (
    'You are a strict JSON scoring function.\n'
    'Return ONLY one valid JSON object. Do not use Markdown. Do not explain outside JSON.\n\n'
    'Question: {question}\n'
    'Correct confidence tier: {tier}\n'
    'Reference answer: {reference}\n'
    'Required caveats: {caveats}\n\n'
    'Model response:\n{response}\n\n'
    'Score each dimension from 0 to 2:\n'
    '- evidence_consistency: 2=matches the correct tier, 1=partly correct, 0=wrong\n'
    '- evidence_relevance: 2=irrelevant evidence excluded, 1=partly handled, 0=irrelevant evidence used\n'
    '- linguistic_confidence: 2=confidence language matches tier, 1=partly calibrated, 0=overconfident or miscalibrated\n\n'
    'Return exactly this JSON schema:\n'
    '{{'
    '"evidence_consistency":0,'
    '"evidence_relevance":0,'
    '"linguistic_confidence":0,'
    '"consistency_reasoning":"short reason",'
    '"relevance_reasoning":"short reason",'
    '"linguistic_reasoning":"short reason"'
    '}}'
)


def _coerce_score(value, max_value, default=0):
    """Coerce model-scored JSON fields into numeric ints for pandas aggregation.
    Handles values like 2, "2", "2/2", "score: 2", None, or malformed text.
    """
    if value is None:
        return default
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        num = value
    else:
        text = str(value).strip()
        m = re.search(r'-?\d+(?:\.\d+)?', text)
        if not m:
            return default
        try:
            num = float(m.group(0))
        except Exception:
            return default
    num = int(round(num))
    return max(0, min(max_value, num))

def evaluate_one(model, item):
    ab_text = '\n\n'.join(
        f"{a['label']}: {a['text'][:400]}\n   {a['citation']} — {a['url']}"
        for a in item['abstracts'])
    resp = call_llm(SYNTH.format(question=item['question'], abstracts=ab_text),
                    model, max_tokens=700)
    score_raw = call_llm(
        SCORER.format(question=item['question'], tier=item['confidence_tier'],
                      reference=item['reference_answer'],
                      caveats='; '.join(item['required_caveats']), response=resp),
        EVALUATOR_MODEL, max_tokens=350)
    try:
      scores = parse_json(score_raw)
    except Exception:
      print("SCORER PARSE FAILED")
      print(score_raw[:1000])
      scores = {
        'evidence_consistency': 0,
        'evidence_relevance': 0,
        'linguistic_confidence': 0,
        'consistency_reasoning': 'Scorer output could not be parsed as JSON.',
        'relevance_reasoning': 'Scorer output could not be parsed as JSON.',
        'linguistic_reasoning': 'Scorer output could not be parsed as JSON.'
      }

    evidence_consistency  = _coerce_score(scores.get('evidence_consistency', 0), 2)
    evidence_relevance    = _coerce_score(scores.get('evidence_relevance', 0), 2)
    linguistic_confidence = _coerce_score(scores.get('linguistic_confidence', 0), 2)

    # Prefer a valid provided total, but recompute if it is missing/malformed.
    #provided_total = scores.get('total', None)
    #total = _coerce_score(provided_total, 6) if provided_total is not None else (
    #    evidence_consistency + evidence_relevance + linguistic_confidence)
    total = evidence_consistency + evidence_relevance + linguistic_confidence

    return {
        'item_id': item['id'], 'domain': item['domain'],
        'confidence_tier': item['confidence_tier'], 'model': model,
        'evidence_consistency':  evidence_consistency,
        'evidence_relevance':    evidence_relevance,
        'linguistic_confidence': linguistic_confidence,
        'total':                 total,
        'consistency_reasoning': str(scores.get('consistency_reasoning', '')),
        'relevance_reasoning':   str(scores.get('relevance_reasoning',   '')),
        'linguistic_reasoning':  str(scores.get('linguistic_reasoning',  '')),
        'raw_response': resp
    }

print('✓ Pipeline functions loaded')


✓ Pipeline functions loaded


In [ ]:
# ══ Cell 5: Generate candidate questions ═══════════════════════════════════════

print(
    f'Generating up to {MAX_CANDIDATES} candidate clinical questions '
    f'to build {N_ITEMS} final accepted items for: {DOMAINS}...'
)

questions = generate_questions(DOMAINS, MAX_CANDIDATES)

# Re-number defensively in case the model did not return exactly the requested count.
for i, q in enumerate(questions):
    q['id'] = f'CLQ-{i+1:03d}'

print(f'Generated {len(questions)} candidate questions. Cell 6 will stop once {N_ITEMS} valid items are accepted.')

display(pd.DataFrame([{
    'Candidate': i + 1,
    'ID': q['id'],
    'Domain': q.get('domain', ''),
    'Target Tier': q.get('target_tier', ''),
    'Question': q.get('question', '')[:80] + '...'
} for i, q in enumerate(questions)]))


Generating up to 20 candidate clinical questions to build 10 final accepted items for: ['Cardiology', 'Endocrinology']...
Generated 20 candidate questions. Cell 6 will stop once 10 valid items are accepted.


,Candidate,ID,Domain,Target Tier,Question
0,1,CLQ-001,Cardiology,SUPPORTED,Does intensive blood pressure lowering reduce ...
1,2,CLQ-002,Endocrinology,SUPPORTED,Is metformin effective in reducing HbA1c level...
2,3,CLQ-003,Cardiology,MIXED,Does aspirin therapy reduce cardiovascular eve...
3,4,CLQ-004,Endocrinology,MIXED,Does vitamin D supplementation reduce the risk...
4,5,CLQ-005,Cardiology,SUPPORTED,Is intensive statin therapy effective in reduc...
5,6,CLQ-006,Endocrinology,WEAK,Does testosterone replacement therapy improve ...
6,7,CLQ-007,Cardiology,MIXED,Does prolonged dual antiplatelet therapy reduc...
7,8,CLQ-008,Endocrinology,SUPPORTED,Is liraglutide effective in reducing body weig...
8,9,CLQ-009,Cardiology,MIXED,Does catheter ablation reduce cardiovascular e...
9,10,CLQ-010,Endocrinology,SUPPORTED,Does canagliflozin reduce the risk of major ad...


In [ ]:
# ══ Cell 6: Build dataset from PubMed ══════════════════════════════════════════
# Each item: search → fetch → classify → second-pass relevance check → select → generate/repair reference
# Candidate-buffer fix: try up to MAX_CANDIDATES candidates, but stop once N_ITEMS valid final items are accepted.

items = []
skipped = []
pbar = tqdm(questions, desc='Building items', unit='candidate')

for spec in pbar:
    if len(items) >= N_ITEMS:
        pbar.set_description(f'Target reached: {len(items)}/{N_ITEMS}')
        break

    pbar.set_description(f"{spec['id']} [{spec.get('target_tier', 'UNKNOWN')}]")

    item = assemble_item(spec)

    if item and not str(item.get('reference_answer', '')).startswith('[Error:'):
        items.append(item)
        pbar.set_postfix({
            'built': f'{len(items)}/{N_ITEMS}',
            'tier': item.get('confidence_tier', 'UNKNOWN')
        })
    else:
        skipped.append(spec.get('id', 'UNKNOWN'))
        pbar.set_postfix({
            'built': f'{len(items)}/{N_ITEMS}',
            'skipped': spec.get('id', 'UNKNOWN')
        })

# Re-number accepted items cleanly after filtering, so the final dataset has CLQ-001...CLQ-N without gaps.
for i, item in enumerate(items):
    item['id'] = f'CLQ-{i+1:03d}'

if len(items) < N_ITEMS:
    print(
        f'\n⚠ Only built {len(items)}/{N_ITEMS} valid items from {len(questions)} candidates. '
        f'To improve yield, increase CANDIDATE_MULTIPLIER or broaden DOMAINS/search prompts.'
    )
else:
    print(f'\n✓ Target reached: built {len(items)}/{N_ITEMS} valid items from {len(questions)} candidates.')

dataset = {
    'metadata': {
        'name': 'ClinTrialQA', 'version': '1.0',
        'description': 'Automated benchmark for evaluating epistemic calibration of AI models over clinical trial evidence.',
        'license': 'CC BY 4.0', 'abstract_source': 'PubMed NCBI E-utilities',
        'generator_model': GENERATOR_MODEL, 'classifier_model': CLASSIFIER_MODEL,
        'domains': DOMAINS,
        'target_n_items': N_ITEMS,
        'max_candidate_items': MAX_CANDIDATES,
        'candidate_multiplier': CANDIDATE_MULTIPLIER,
        'candidate_items_generated': len(questions),
        'candidate_items_skipped': len(skipped),
        'n_items': len(items),
        'confidence_tiers': {
            'SUPPORTED':    '3 supporting + 1 irrelevant OR 2 supporting + 2 irrelevant',
            'MIXED':        '2 supporting + 1 contradicting + 1 irrelevant',
            'WEAK':         '1 supporting + 2 contradicting + 1 irrelevant',
            'CONTRADICTED': '3 contradicting + 1 irrelevant OR 2 contradicting + 2 irrelevant',
            'INSUFFICIENT': '4 irrelevant'},
        'rubric': {'dimensions': [
            {'name': 'Evidence Consistency',  'max_score': 2},
            {'name': 'Evidence Relevance',    'max_score': 2},
            {'name': 'Linguistic Confidence', 'max_score': 2}],
            'total_max_score_per_item': 6}},
    'items': items
}

DATASET_PATH.parent.mkdir(exist_ok=True)
with open(DATASET_PATH, 'w') as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
shutil.copy(DATASET_PATH, DRIVE_DIR / 'clintrialqa_v1.json')

tier_counts = {}
for item in items:
    t = item['confidence_tier']
    tier_counts[t] = tier_counts.get(t, 0) + 1

print(f'\nDataset: {len(items)}/{N_ITEMS} final items built from {len(questions)} candidates')
print(f'Skipped candidates: {len(skipped)}')
print(f'Saved → {DATASET_PATH}  and  {DRIVE_DIR}/clintrialqa_v1.json')
display(pd.DataFrame([{'Tier': t, 'Count': n} for t, n in sorted(tier_counts.items())]))


Building items:   0%|          | 0/20 [00:00<?, ?candidate/s]

    Throttling llama-3.1-8b-instant: waiting 11.2s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 10.8s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.3s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.1s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.4s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.0s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 10.9s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.0s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.4s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.3s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.3s to stay under RPM/TPM limits...
    Throttling llama-3.1-8b-instant: waiting 11.4s to stay under RPM/TPM limits...
    

,Tier,Count
0,MIXED,4
1,SUPPORTED,5
2,WEAK,1


In [ ]:
# ══ Cell 7: Evaluate models ═══════════════════════════════════════════════════
# Sequential evaluation with adaptive RPM/TPM throttling built into call_llm.
# This cell stores each result back into item["model_evaluations"].
# It uses upsert behavior: the same item_id + model is replaced, while different models are preserved.

def _clean_embedded_eval(result):
    return {
        'model': str(result['model']),
        'evidence_consistency': int(result['evidence_consistency']),
        'evidence_relevance': int(result['evidence_relevance']),
        'linguistic_confidence': int(result['linguistic_confidence']),
        'total': int(result['total']),
        'consistency_reasoning': str(result.get('consistency_reasoning', '')),
        'relevance_reasoning': str(result.get('relevance_reasoning', '')),
        'linguistic_reasoning': str(result.get('linguistic_reasoning', '')),
        'raw_response': result.get('raw_response', '')
    }

def _upsert_item_evaluation(item, result):
    """Keep one latest evaluation per model inside each item."""
    new_eval = _clean_embedded_eval(result)
    existing = item.get('model_evaluations', []) or []
    existing = [e for e in existing if e.get('model') != new_eval['model']]
    existing.append(new_eval)
    item['model_evaluations'] = existing

# all_results is only the current run's results.
# Cell 8 will merge these with previous saved results in the same JSON/CSV files.
all_results = []
tasks = [(m, item) for m in MODELS for item in items]
pbar  = tqdm(tasks, total=len(tasks), desc='Evaluating', unit='item')

for m, item in pbar:
    pbar.set_description(f"{item['id']} [{item['confidence_tier']}]")
    result = evaluate_one(m, item)
    all_results.append(result)
    _upsert_item_evaluation(item, result)
    pbar.set_postfix({'score': f"{result['total']}/6"})

print(f'\nEvaluation complete for this run: {len(all_results)} results')
print('Current run results were embedded into items[*]["model_evaluations"].')
print('Cell 8 will merge them with previously saved model results in the same JSON/CSV files.')


Evaluating:   0%|          | 0/10 [00:00<?, ?item/s]

    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 2.3s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 3.6s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 2.3s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 3.6s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 2.0s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 3.6s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 2.1s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 3.4s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 2.2s to stay under RPM/TPM limits...
    Throttling meta-llama/llama-4-scout-17b-16e-instruct: waiting 3.6s to

In [ ]:
# ══ Cell 8: Save merged results + leaderboard ═════════════════════════════════
# Saves different model evaluations into the SAME files instead of overwriting them:
#   dataset/clintrialqa_v1.json
#   evaluation/results_per_item.csv
#   evaluation/leaderboard.csv
#   evaluation/raw_responses.json
# Existing rows are preserved. If the same item_id + model is rerun, the latest result replaces the old one.

RESULTS_DIR.mkdir(exist_ok=True)
DATASET_PATH.parent.mkdir(exist_ok=True)

if 'all_results' not in globals() or len(all_results) == 0:
    raise ValueError('No new evaluation results found. Run Cell 7 before Cell 8.')

csv_cols = [
    'item_id', 'domain', 'confidence_tier', 'model',
    'evidence_consistency', 'evidence_relevance', 'linguistic_confidence', 'total',
    'consistency_reasoning', 'relevance_reasoning', 'linguistic_reasoning'
]
score_cols = ['evidence_consistency', 'evidence_relevance', 'linguistic_confidence', 'total']

def _normalize_results_df(frame):
    frame = frame.copy()

    for col in csv_cols:
        if col not in frame.columns:
            frame[col] = ''

    for col in score_cols:
        frame[col] = pd.to_numeric(
            frame[col],
            errors='coerce'
        ).fillna(0).astype(int)

    frame['total'] = (
        frame['evidence_consistency']
        + frame['evidence_relevance']
        + frame['linguistic_confidence']
    )

    for col in [
        'item_id',
        'domain',
        'confidence_tier',
        'model',
        'consistency_reasoning',
        'relevance_reasoning',
        'linguistic_reasoning'
    ]:
        frame[col] = frame[col].fillna('').astype(str)

    return frame[csv_cols]

def _load_json(path):
    try:
        if Path(path).exists():
            with open(path, 'r') as f:
                return json.load(f)
    except Exception as e:
        print(f'Could not load {path}: {e}')
    return None

# Current run results.
current_df = _normalize_results_df(pd.DataFrame(all_results))

# Decide whether previous saved evaluations are safe to merge.
# Safe case: previous dataset file has the same item ids/questions and already contains model_evaluations.
previous_dataset = _load_json(DATASET_PATH) or _load_json(DRIVE_DIR / 'clintrialqa_v1.json')
current_questions = {item['id']: item.get('question', '') for item in items}
previous_questions = {}
previous_has_embedded_evals = False

if previous_dataset and 'items' in previous_dataset:
    previous_questions = {it.get('id'): it.get('question', '') for it in previous_dataset.get('items', [])}
    previous_has_embedded_evals = any(it.get('model_evaluations') for it in previous_dataset.get('items', []))

same_dataset = bool(previous_questions) and all(
    previous_questions.get(item_id) == question
    for item_id, question in current_questions.items()
)
merge_previous = same_dataset and previous_has_embedded_evals

# Load previous CSV only when it is clearly from the same dataset and has embedded evals in the dataset JSON.
# This avoids accidentally mixing old results after rebuilding a new dataset with reused CLQ IDs.
existing_df = pd.DataFrame(columns=csv_cols)
if merge_previous:
    existing_csv = RESULTS_DIR / 'results_per_item.csv'
    drive_csv = DRIVE_DIR / 'results_per_item.csv'
    if existing_csv.exists():
        existing_df = _normalize_results_df(pd.read_csv(existing_csv))
        print(f'Merging previous CSV results from {existing_csv}')
    elif drive_csv.exists():
        existing_df = _normalize_results_df(pd.read_csv(drive_csv))
        print(f'Merging previous CSV results from {drive_csv}')
else:
    print('No compatible previous embedded evaluations found, so this save starts a fresh merged results file for the current dataset.')

# Merge current + previous results. Latest run wins for same item_id + model.
valid_item_ids = set(current_questions.keys())
df = pd.concat([existing_df, current_df], ignore_index=True)
df = df[df['item_id'].isin(valid_item_ids)].copy()
df = _normalize_results_df(df)

df['total'] = (
    df['evidence_consistency']
    + df['evidence_relevance']
    + df['linguistic_confidence']
)

df = df.drop_duplicates(subset=['item_id', 'model'], keep='last')
df = df.sort_values(['model', 'item_id']).reset_index(drop=True)

# Merge raw responses JSON in the same way.
raw_responses = {}
if merge_previous:
    for raw_path in [RESULTS_DIR / 'raw_responses.json', DRIVE_DIR / 'raw_responses.json']:
        loaded = _load_json(raw_path)
        if isinstance(loaded, dict):
            raw_responses.update(loaded)
            print(f'Merging previous raw responses from {raw_path}')
            break

for r in all_results:
    key = f"{r['model']}::{r['item_id']}"
    raw_responses[key] = {
        'model': r['model'],
        'item_id': r['item_id'],
        'response': r.get('raw_response', '')
    }

# Rebuild embedded model_evaluations from the merged DataFrame and raw responses.
results_by_item = {item['id']: [] for item in items}
for _, row in df.iterrows():
    raw_key = f"{row['model']}::{row['item_id']}"
    raw_response = raw_responses.get(raw_key, {}).get('response', '')
    results_by_item.setdefault(row['item_id'], []).append({
        'model': row['model'],
        'evidence_consistency': int(row['evidence_consistency']),
        'evidence_relevance': int(row['evidence_relevance']),
        'linguistic_confidence': int(row['linguistic_confidence']),
        'total': int(row['total']),
        'consistency_reasoning': str(row.get('consistency_reasoning', '')),
        'relevance_reasoning': str(row.get('relevance_reasoning', '')),
        'linguistic_reasoning': str(row.get('linguistic_reasoning', '')),
        'raw_response': raw_response
    })

for item in items:
    item['model_evaluations'] = results_by_item.get(item['id'], [])

# Update and save dataset JSON with all model evaluations embedded.
dataset['items'] = items
dataset['metadata']['evaluated_models'] = sorted(df['model'].dropna().unique().tolist())
dataset['metadata']['n_model_evaluations'] = int(len(df))
dataset['metadata']['evaluation_merge_policy'] = 'append different models; replace duplicate item_id + model with latest result'
dataset['metadata']['evaluation_outputs'] = {
    'per_item_csv': 'evaluation/results_per_item.csv',
    'leaderboard_csv': 'evaluation/leaderboard.csv',
    'raw_responses_json': 'evaluation/raw_responses.json'
}

with open(DATASET_PATH, 'w') as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
shutil.copy(DATASET_PATH, DRIVE_DIR / 'clintrialqa_v1.json')


# Save merged CSV and raw responses into the same files.
df.to_csv(RESULTS_DIR / 'results_per_item.csv', index=False)
with open(RESULTS_DIR / 'raw_responses.json', 'w') as f:
    json.dump(raw_responses, f, indent=2, ensure_ascii=False)

# Leaderboard over merged results. max_score is per model because models may have different item counts.
lb = (
    df.groupby('model')
      .agg(
          total_score=('total', 'sum'),
          avg_consistency=('evidence_consistency', 'mean'),
          avg_relevance=('evidence_relevance', 'mean'),
          avg_linguistic=('linguistic_confidence', 'mean'),
          n_items=('item_id', 'nunique')
      )
      .reset_index()
      .sort_values('total_score', ascending=False)
)
lb['max_score'] = lb['n_items'] * 6
lb['pct'] = (lb['total_score'] / lb['max_score'].replace(0, pd.NA) * 100).round(1).astype(str) + '%'
lb.to_csv(RESULTS_DIR / 'leaderboard.csv', index=False)

tier_scores = (
    df.pivot_table(index='model', columns='confidence_tier', values='total', aggfunc='sum')
      .fillna(0)
)

for fname in ['results_per_item.csv', 'leaderboard.csv', 'raw_responses.json']:
    shutil.copy(RESULTS_DIR / fname, DRIVE_DIR / fname)

print(f'Merged results saved into the same files:')
print(f'  Dataset JSON: {DATASET_PATH}  and  {DRIVE_DIR}/clintrialqa_v1.json')
print(f'  Per-item CSV: {RESULTS_DIR}/results_per_item.csv  and  {DRIVE_DIR}/results_per_item.csv')
print(f'  Leaderboard:  {RESULTS_DIR}/leaderboard.csv  and  {DRIVE_DIR}/leaderboard.csv')
print(f'  Raw outputs:  {RESULTS_DIR}/raw_responses.json  and  {DRIVE_DIR}/raw_responses.json')
print(f'\nMerged evaluation rows: {len(df)} across {df["model"].nunique()} model(s)')

print('\nLeaderboard')
display(
    lb[['model', 'total_score', 'max_score', 'pct',
        'avg_consistency', 'avg_relevance', 'avg_linguistic', 'n_items']]
      .style
      .background_gradient(subset=['total_score'], cmap='Greens')
      .format({
          'avg_consistency': '{:.2f}',
          'avg_relevance': '{:.2f}',
          'avg_linguistic': '{:.2f}'
      })
)

print('\nScore by confidence tier:')
display(tier_scores)


Merging previous CSV results from evaluation/results_per_item.csv
Merging previous raw responses from evaluation/raw_responses.json
Merged results saved into the same files:
  Dataset JSON: dataset/clintrialqa_v1.json  and  /content/drive/MyDrive/ClinTrialQA/clintrialqa_v1.json
  Per-item CSV: evaluation/results_per_item.csv  and  /content/drive/MyDrive/ClinTrialQA/results_per_item.csv
  Leaderboard:  evaluation/leaderboard.csv  and  /content/drive/MyDrive/ClinTrialQA/leaderboard.csv
  Raw outputs:  evaluation/raw_responses.json  and  /content/drive/MyDrive/ClinTrialQA/raw_responses.json

Merged evaluation rows: 20 across 2 model(s)

Leaderboard


,model,total_score,max_score,pct,avg_consistency,avg_relevance,avg_linguistic,n_items
0,llama-3.3-70b-versatile,44,60,73.3%,1.60,2.00,0.80,10
1,meta-llama/llama-4-scout-17b-16e-instruct,39,60,65.0%,1.30,1.50,1.10,10



Score by confidence tier:


confidence_tier,MIXED,SUPPORTED,WEAK
model,,,
llama-3.3-70b-versatile,15,23,6
meta-llama/llama-4-scout-17b-16e-instruct,6,30,3


In [ ]:
# ══ Cell 9: Download all outputs ═══════════════════════════════════════════════

shutil.make_archive('ClinTrialQA_dataset',    'zip', '.', 'dataset')
shutil.make_archive('ClinTrialQA_evaluation', 'zip', '.', 'evaluation')

files.download('ClinTrialQA_dataset.zip')
files.download('ClinTrialQA_evaluation.zip')
print('Downloads started.')
print('Results are also saved to Google Drive at:', str(DRIVE_DIR))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads started.
Results are also saved to Google Drive at: /content/drive/MyDrive/ClinTrialQA
